# 1. Introdução e desenho experimental

Este notebook reconstrói, de forma didática e reproduzível, a modelagem do Capítulo 4 do TCC/MBA sobre predição de lead time portuário. A variável alvo é `t_total_port_stay_h`, isto é, o tempo total de permanência da embarcação no porto, em horas.

A pergunta operacional é: no instante da chegada ao porto (`arrival_port_ts`), quantas horas a embarcação deve permanecer até a saída? Como a previsão simula uma aplicação ao longo do tempo, o desenho experimental usa divisão temporal, e não split aleatório.

O risco metodológico central é o vazamento temporal. Uma feature só pode entrar no modelo se estiver disponível no instante da previsão ou puder ser calculada apenas com informação já conhecida. Por isso, as features históricas reconstruídas usam cutoff conservador: para uma chegada no dia D, apenas informações conhecidas até 23:59:59 de D-1 podem ser usadas.

Os conjuntos têm papéis distintos: `train` ajusta os modelos; `validation` seleciona modelo, features e hiperparâmetros; `calibration` funciona como verificação secundária; `final_test` permanece fechado até a especificação estar congelada.

# 2. Imports, configuração e reprodutibilidade

Esta seção define constantes, caminhos e bibliotecas. O notebook começa em `data/processed/eda_base.parquet` e salva artefatos próprios em `results/cap4_notebook_run/`, preservando os resultados oficiais em `results/cap4_rebuild/`.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import platform
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_pinball_loss, mean_squared_error, median_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.3f}".format)

In [ ]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Raiz do projeto não encontrada.")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features.historical_context import add_cyclical_features, build_historical_features, run_leakage_tests

DATA_FILE = ROOT / "data" / "processed" / "eda_base.parquet"
RUN_RESULTS = ROOT / "results" / "cap4_notebook_run"
RUN_RESULTS.mkdir(parents=True, exist_ok=True)

TARGET = "t_total_port_stay_h"
DATE_COL = "arrival_port_ts"
PORT_COL = "port"
OPERATION_COL = "operation_type"
RANDOM_STATE = 42
MIN_GROUP_SIZE = 30
SPLIT_BOUNDS = {
    "train_start": "2023-01-01",
    "validation_start": "2024-07-01",
    "calibration_start": "2025-01-01",
    "final_test_start": "2025-07-01",
    "final_test_end": "2026-01-01",
}

In [ ]:
def git_value(args: list[str]) -> str:
    try:
        return subprocess.check_output(["git", *args], cwd=ROOT, text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return "indisponível"

repro_context = pd.DataFrame(
    {
        "item": ["root", "branch", "commit", "python", "platform"],
        "valor": [str(ROOT), git_value(["branch", "--show-current"]), git_value(["rev-parse", "HEAD"]), platform.python_version(), platform.platform()],
    }
)
repro_context

# 3. Carregamento da base de modelagem

A base de partida da modelagem é `eda_base.parquet`. Ela contém a base analítica consolidada, mas as features históricas temporalmente seguras serão reconstruídas explicitamente mais adiante neste notebook.

In [ ]:
df_raw = pd.read_parquet(DATA_FILE)
for col in [DATE_COL, "berthing_ts", "unberthing_ts", "departure_port_ts"]:
    if col in df_raw.columns:
        df_raw[col] = pd.to_datetime(df_raw[col], errors="coerce")

base_summary = pd.DataFrame(
    {
        "item": ["registros", "colunas", "menor chegada", "maior chegada", "target", "target médio (h)", "target mediano (h)"],
        "valor": [len(df_raw), df_raw.shape[1], df_raw[DATE_COL].min(), df_raw[DATE_COL].max(), TARGET, df_raw[TARGET].mean(), df_raw[TARGET].median()],
    }
)
base_summary

In [ ]:
important_cols = [
    "port_call_id", PORT_COL, "port_display", OPERATION_COL, "vessel_id", "source_port", "destination_port",
    DATE_COL, "berthing_ts", "unberthing_ts", "departure_port_ts", TARGET,
]
df_raw[[col for col in important_cols if col in df_raw.columns]].head()

In [ ]:
df_raw[TARGET].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("horas")

# 4. Features originais do Capítulo 4

O conjunto `ORIGINAL` reproduz as variáveis usadas antes do enriquecimento histórico seguro. Elas são separadas em famílias para deixar claro o papel de cada grupo: estrutura/geografia, calendário, clima histórico defasado e flags operacionais declaradas.

In [ ]:
def chapter4_feature_lists(df: pd.DataFrame) -> tuple[list[str], list[str], list[str]]:
    categorical = ["port", "region", "state", "arrival_shift", "arrival_season"]
    calendar = [
        "arrival_quarter", "arrival_weekofyear", "arrival_is_weekend",
        "arrival_hour_sin", "arrival_hour_cos", "arrival_dow_sin", "arrival_dow_cos",
        "arrival_month_sin", "arrival_month_cos",
    ]
    weather = [
        "rain_sum_prev_1d", "precipitation_sum_prev_1d", "wind_speed_10m_max_prev_1d",
        "wind_gusts_10m_max_prev_1d", "temperature_2m_mean_prev_1d",
        "rain_sum_prev_3d", "precipitation_hours_prev_3d", "temperature_2m_mean_prev_3d",
        "wind_speed_10m_max_prev_3d", "wind_gusts_10m_max_prev_3d",
        "rain_sum_prev_7d", "precipitation_hours_prev_7d", "temperature_2m_mean_prev_7d",
        "wind_speed_10m_max_prev_7d", "wind_gusts_10m_max_prev_7d",
    ]
    operation = sorted(col for col in df.columns if col.startswith("op_"))
    categorical = [col for col in categorical if col in df.columns]
    numerical = [col for col in calendar + weather if col in df.columns]
    operation = [col for col in operation if col in df.columns]
    return categorical, numerical, operation

feature_family_original = {
    "estruturais_geograficas": ["port", "region", "state"],
    "temporais_calendario": [
        "arrival_shift", "arrival_season", "arrival_quarter", "arrival_weekofyear", "arrival_is_weekend",
        "arrival_hour_sin", "arrival_hour_cos", "arrival_dow_sin", "arrival_dow_cos", "arrival_month_sin", "arrival_month_cos",
    ],
    "clima_historico_defasado": [col for col in df_raw.columns if col.endswith(("_prev_1d", "_prev_3d", "_prev_7d")) 
                                 and col.startswith(("rain_", "precipitation_", "wind_", "temperature_"))],
    "flags_operacionais": sorted(col for col in df_raw.columns if col.startswith("op_")),
}

original_family_table = pd.DataFrame(
    [{"familia": family, "n_features": len(cols), "exemplos": ", ".join(cols[:8])} for family, cols in feature_family_original.items()]
)
original_family_table

# 5. Construção das variáveis temporais/cíclicas

Hora, dia da semana e mês são variáveis cíclicas: 23h e 0h estão próximas, assim como dezembro e janeiro. A transformação seno/cosseno representa essa circularidade sem criar uma ruptura artificial nas fronteiras do calendário.

In [ ]:
df = df_raw.dropna(subset=[DATE_COL, TARGET]).copy()
df = df[(df[DATE_COL] >= SPLIT_BOUNDS["train_start"]) & (df[DATE_COL] < SPLIT_BOUNDS["final_test_end"])].copy()
df = add_cyclical_features(df)

cyclical_examples = ["arrival_hour", "arrival_hour_sin", "arrival_hour_cos", "arrival_dayofweek", "arrival_dow_sin", 
                     "arrival_dow_cos", "arrival_month", "arrival_month_sin", "arrival_month_cos"]
df[[col for col in cyclical_examples if col in df.columns]].head(10)

# 8. Reconstrução das features históricas temporalmente seguras

Nesta etapa o notebook executa de fato a reconstrução das features históricas. A diferença em relação ao conjunto `ORIGINAL` é que `ENRICHED_SAFE_HISTORY` incorpora memória operacional e estatísticas conhecidas antes do cutoff de cada escala.

A regra temporal é conservadora: para prever uma escala, somente informação conhecida antes do cutoff definido para ela pode participar das features históricas. Durações de espera, operação e permanência total só entram no histórico depois dos eventos que as tornam conhecidas.

In [ ]:
@dataclass(frozen=True)
class FeatureRegistry:
    original_features: list[str]
    enriched_features: list[str]
    categorical_features: list[str]
    numerical_features: list[str]
    operation_features: list[str]
    historical_families: dict[str, list[str]]
    feature_metadata: dict[str, str]

categorical_features, numerical_features, operation_features = chapter4_feature_lists(df)
original_features = categorical_features + numerical_features + operation_features

historical = build_historical_features(df, operation_features)
df_model = historical.data.copy()
family_order = ["flow", "port_performance", "state", "vessel_history", "vessel_port_history", "source_history", "destination_history", 
                "route_history", "operation_mix"]
historical_cols = []
for family in family_order:
    historical_cols.extend(historical.families.get(family, []))

enriched_features = list(dict.fromkeys(original_features + historical_cols))
registry = FeatureRegistry(original_features, enriched_features, categorical_features, numerical_features, operation_features, 
                           historical.families, historical.metadata)

pd.DataFrame(
    [{"familia": family, "n_features": len(cols), "exemplos": ", ".join(cols[:6])} for family, cols in historical.families.items()]
)

In [ ]:
sample_historical_cols = [
    "cutoff_ts", "arrivals_prev_1d", "departures_prev_1d", "flow_balance_prev_1d",
    "state_waiting_vessels_d_minus_1", "port_total_known_median", "vessel_total_known_median",
    "vessel_port_total_known_median", "source_total_known_median", "destination_total_known_median", "route_total_known_median",
]
shown_cols = [col for col in ["port_call_id", PORT_COL, OPERATION_COL, DATE_COL, TARGET] + sample_historical_cols if col in df_model.columns]
df_model[shown_cols].head(12)

# 9. Testes de leakage temporal

Os testes adversariais verificam se a reconstrução impede uso do próprio evento, eventos futuros, dias sem movimento mal tratados e estatísticas conhecidas somente após berthing, unberthing ou departure.

In [ ]:
leakage_tests = run_leakage_tests()
leakage_tests.to_csv(RUN_RESULTS / "leakage_tests.csv", index=False)
leakage_tests

In [ ]:
if not leakage_tests["passed"].all():
    raise RuntimeError("Pelo menos um teste de leakage falhou. Interrompa a análise antes de modelar.")
print("Todos os testes de leakage passaram.")

# 10. Divisão temporal

Os splits são criados programaticamente com os limites oficiais. Até este ponto, `final_test` não participa de seleção de modelo ou hiperparâmetro.

In [ ]:
def split_data(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    return {
        "train": df[(df[DATE_COL] >= SPLIT_BOUNDS["train_start"]) & (df[DATE_COL] < SPLIT_BOUNDS["validation_start"])].copy(),
        "validation": df[(df[DATE_COL] >= SPLIT_BOUNDS["validation_start"]) & (df[DATE_COL] < SPLIT_BOUNDS["calibration_start"])].copy(),
        "calibration": df[(df[DATE_COL] >= SPLIT_BOUNDS["calibration_start"]) & (df[DATE_COL] < SPLIT_BOUNDS["final_test_start"])].copy(),
        "final_test": df[(df[DATE_COL] >= SPLIT_BOUNDS["final_test_start"]) & (df[DATE_COL] < SPLIT_BOUNDS["final_test_end"])].copy(),
    }

splits = split_data(df_model)
split_summary = pd.DataFrame(
    [
        {
            "dataset": name,
            "primeira_data": part[DATE_COL].min(),
            "ultima_data": part[DATE_COL].max(),
            "n_observacoes": len(part),
            "target_medio_h": part[TARGET].mean(),
            "target_mediano_h": part[TARGET].median(),
        }
        for name, part in splits.items()
    ]
)
split_summary.to_csv(RUN_RESULTS / "split_summary.csv", index=False)
split_summary

# 11. Métricas de avaliação

O MAE é a métrica principal porque mede erro médio em horas, diretamente interpretável para a decisão operacional. RMSE penaliza erros grandes, MedAE descreve erro típico robusto e RMSLE reduz o peso relativo da longa cauda. As métricas Q90/Q95 avaliam o comportamento nos casos extremos, importantes para estoque de segurança.

In [ ]:
def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.clip(np.asarray(y_true, dtype=float), 0, None)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "medae": float(median_absolute_error(y_true, y_pred)),
        "rmsle": rmsle(y_true, y_pred),
    }


def tail_metrics(y_true: np.ndarray, y_pred: np.ndarray, q90_threshold: float, q95_threshold: float) -> dict[str, float | int]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    out = {"q90_threshold_h": float(q90_threshold), "q95_threshold_h": float(q95_threshold)}
    for label, threshold in [("q90", q90_threshold), ("q95", q95_threshold)]:
        mask = y_true >= threshold
        errors = y_pred[mask] - y_true[mask]
        out[f"n_{label}"] = int(mask.sum())
        out[f"mae_{label}"] = float(np.abs(errors).mean()) if mask.any() else np.nan
        out[f"bias_{label}"] = float(errors.mean()) if mask.any() else np.nan
        out[f"underprediction_rate_{label}"] = float((errors < 0).mean()) if mask.any() else np.nan
    return out


def score_row(model: str, feature_config: str, dataset: str, y_true: np.ndarray, y_pred: np.ndarray, q90: float, q95: float, 
              extra: dict[str, object] | None = None) -> dict[str, object]:
    row = {"model": model, "feature_config": feature_config, "dataset": dataset, **regression_metrics(y_true, y_pred), 
           **tail_metrics(y_true, y_pred, q90, q95)}
    if extra:
        row.update(extra)
    return row

# 12. Baselines históricos

Os baselines representam políticas históricas simples calculadas apenas sobre o treino: mediana global, mediana por porto e mediana hierárquica porto × tipo de operação com fallback.

In [ ]:
def group_quantile_prediction(train: pd.DataFrame, scoring: pd.DataFrame, group_cols: list[str], quantile: float, min_group_size: int = 1) -> np.ndarray:
    global_q = train[TARGET].quantile(quantile)
    stats = train.groupby(group_cols, dropna=False)[TARGET].agg(q=lambda x: x.quantile(quantile), count="count").reset_index()
    stats.loc[stats["count"] < min_group_size, "q"] = np.nan
    scored = scoring[group_cols].copy().merge(stats[group_cols + ["q"]], on=group_cols, how="left")
    return scored["q"].fillna(global_q).to_numpy(dtype=float)


def hierarchical_quantile_prediction(train: pd.DataFrame, scoring: pd.DataFrame, quantile: float, primary_cols: list[str] | None = None, fallback_cols: list[str] | None = None, min_group_size: int = MIN_GROUP_SIZE) -> np.ndarray:
    primary_cols = primary_cols or [PORT_COL, OPERATION_COL]
    fallback_cols = fallback_cols or [PORT_COL]
    global_q = train[TARGET].quantile(quantile)
    primary = train.groupby(primary_cols, dropna=False)[TARGET].agg(q=lambda x: x.quantile(quantile), count="count").reset_index()
    primary = primary[primary["count"] >= min_group_size][primary_cols + ["q"]].rename(columns={"q": "primary_q"})
    fallback = train.groupby(fallback_cols, dropna=False)[TARGET].agg(q=lambda x: x.quantile(quantile), count="count").reset_index()
    fallback = fallback[fallback["count"] >= min_group_size][fallback_cols + ["q"]].rename(columns={"q": "fallback_q"})
    out = scoring[primary_cols].copy().merge(primary, on=primary_cols, how="left").merge(fallback, on=fallback_cols, how="left")
    return out["primary_q"].fillna(out["fallback_q"]).fillna(global_q).to_numpy(dtype=float)

train = splits["train"]
validation = splits["validation"]
calibration = splits["calibration"]
q90_train = float(train[TARGET].quantile(0.90))
q95_train = float(train[TARGET].quantile(0.95))

baseline_predictions = {
    "baseline_global_median": np.repeat(train[TARGET].median(), len(validation)),
    "baseline_median_by_port": group_quantile_prediction(train, validation, [PORT_COL], 0.50),
    "baseline_hierarchical_port_operation": hierarchical_quantile_prediction(train, validation, 0.50),
}
baseline_validation = pd.DataFrame([
    score_row(name, "ORIGINAL", "validation", validation[TARGET], pred, q90_train, q95_train)
    for name, pred in baseline_predictions.items()
])
baseline_validation.to_csv(RUN_RESULTS / "baseline_validation.csv", index=False)
baseline_validation

# 13. Funções auxiliares de preprocessing

Ridge, Random Forest e Gradient Boosting usam imputação para numéricas, one-hot encoding para categóricas e scaling apenas no Ridge. O HistGradientBoosting recebe colunas categóricas como `category`, aprendendo categorias somente a partir do treino.

In [ ]:
def build_preprocessor(numeric_features: list[str], categorical_features: list[str], scale_numeric: bool = False) -> ColumnTransformer:
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", Pipeline(numeric_steps), numeric_features), ("cat", categorical_pipeline, categorical_features)])


def prepare_hgb_frames(train_df: pd.DataFrame, eval_df: pd.DataFrame, features: list[str], categorical_features: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    x_train = train_df[features].copy()
    x_eval = eval_df[features].copy()
    for col in categorical_features:
        if col not in features:
            continue
        categories = pd.Index(x_train[col].astype("string").dropna().astype(str).unique())
        x_train[col] = pd.Categorical(x_train[col].astype("string"), categories=categories)
        x_eval[col] = pd.Categorical(x_eval[col].astype("string"), categories=categories)
    return x_train, x_eval


def append_eval_scores(rows: list[dict[str, object]], label: str, feature_config: str, pred_eval: np.ndarray, extra: dict[str, object] | None = None) -> None:
    n_val = len(validation)
    for dataset, part, pred in [("validation", validation, pred_eval[:n_val]), ("calibration", calibration, pred_eval[n_val:])]:
        rows.append(score_row(label, feature_config, dataset, part[TARGET], pred, q90_train, q95_train, extra=extra))

# 14. Transformação do target

Os modelos pontuais são treinados com `np.log1p(y_train)` e as previsões retornam para horas com `np.expm1(...)`, truncadas em zero. Essa transformação reduz a influência da assimetria e da longa cauda sem mudar a unidade de interpretação dos resultados finais.

In [ ]:
y_train_log = np.log1p(train[TARGET].to_numpy(dtype=float))
pd.DataFrame({"target_h": train[TARGET].head(5).to_numpy(), "log1p_target": y_train_log[:5]})

# 15. Ridge Regression

O Ridge é treinado com target log-transformado. A busca de `alpha` usa exatamente os candidatos da execução canônica: `0.01`, `0.1`, `1.0`, `10.0` e `100.0`, sempre selecionando pelo desempenho em `validation`.

In [ ]:
def tune_ridge_alpha_visible(features: list[str], config_name: str) -> tuple[float, pd.DataFrame]:
    rows = []
    numeric_features = [col for col in features if col not in categorical_features]
    for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:
        candidate_ridge = Pipeline([
            ("preprocessor", build_preprocessor(numeric_features, [col for col in categorical_features if col in features], True)),
            ("model", Ridge(alpha=float(alpha))),
        ])
        candidate_ridge.fit(train[features], np.log1p(train[TARGET].to_numpy(dtype=float)))
        pred_validation = np.clip(np.expm1(candidate_ridge.predict(validation[features])), 0, None)
        rows.append({"feature_config": config_name, "alpha": alpha, **regression_metrics(validation[TARGET], pred_validation)})
    table = pd.DataFrame(rows).sort_values(["mae", "rmse"]).reset_index(drop=True)
    return float(table.iloc[0]["alpha"]), table

ridge_alpha_original, ridge_alpha_table_original = tune_ridge_alpha_visible(original_features, "ORIGINAL")
ridge_alpha_enriched, ridge_alpha_table_enriched = tune_ridge_alpha_visible(enriched_features, "ENRICHED_SAFE_HISTORY")
ridge_alpha_table = pd.concat([ridge_alpha_table_original, ridge_alpha_table_enriched], ignore_index=True)
ridge_alpha_table.to_csv(RUN_RESULTS / "ridge_alpha_validation.csv", index=False)
ridge_alpha_table

In [ ]:
eval_model = pd.concat([validation, calibration], ignore_index=True)
point_rows = []
point_models = {}
point_predictions_eval = {}

ridge_original_features = original_features
ridge_original_numeric = [col for col in ridge_original_features if col not in categorical_features]
ridge_original_model = Pipeline([
    ("preprocessor", build_preprocessor(ridge_original_numeric, [col for col in categorical_features if col in ridge_original_features], True)),
    ("model", Ridge(alpha=ridge_alpha_original)),
])
ridge_original_model.fit(train[ridge_original_features], np.log1p(train[TARGET].to_numpy(dtype=float)))
ridge_original_pred_eval = np.clip(np.expm1(ridge_original_model.predict(eval_model[ridge_original_features])), 0, None)
point_models[("Ridge", "ORIGINAL")] = ridge_original_model
point_predictions_eval[("Ridge", "ORIGINAL")] = ridge_original_pred_eval
append_eval_scores(point_rows, "Ridge", "ORIGINAL", ridge_original_pred_eval, {"ridge_alpha": ridge_alpha_original})

ridge_enriched_features = enriched_features
ridge_enriched_numeric = [col for col in ridge_enriched_features if col not in categorical_features]
ridge_enriched_model = Pipeline([
    ("preprocessor", build_preprocessor(ridge_enriched_numeric, [col for col in categorical_features if col in ridge_enriched_features], True)),
    ("model", Ridge(alpha=ridge_alpha_enriched)),
])
ridge_enriched_model.fit(train[ridge_enriched_features], np.log1p(train[TARGET].to_numpy(dtype=float)))
ridge_enriched_pred_eval = np.clip(np.expm1(ridge_enriched_model.predict(eval_model[ridge_enriched_features])), 0, None)
point_models[("Ridge", "ENRICHED_SAFE_HISTORY")] = ridge_enriched_model
point_predictions_eval[("Ridge", "ENRICHED_SAFE_HISTORY")] = ridge_enriched_pred_eval
append_eval_scores(point_rows, "Ridge", "ENRICHED_SAFE_HISTORY", ridge_enriched_pred_eval, {"ridge_alpha": ridge_alpha_enriched})

pd.DataFrame([row for row in point_rows if row["model"] == "Ridge"])

# 16. Random Forest

O Random Forest usa a configuração canônica: `n_estimators=250`, `min_samples_leaf=50`, `max_features=1.0`, `random_state=42` e `n_jobs=-1`.

In [ ]:
rf_original_features = original_features
rf_original_numeric = [col for col in rf_original_features if col not in categorical_features]
rf_original_model = Pipeline([
    ("preprocessor", build_preprocessor(rf_original_numeric, [col for col in categorical_features if col in rf_original_features], False)),
    ("model", RandomForestRegressor(n_estimators=250, min_samples_leaf=50, max_features=1.0, random_state=RANDOM_STATE, n_jobs=-1)),
])
rf_original_model.fit(train[rf_original_features], np.log1p(train[TARGET].to_numpy(dtype=float)))
rf_original_pred_eval = np.clip(np.expm1(rf_original_model.predict(eval_model[rf_original_features])), 0, None)
point_models[("Random Forest", "ORIGINAL")] = rf_original_model
point_predictions_eval[("Random Forest", "ORIGINAL")] = rf_original_pred_eval
append_eval_scores(point_rows, "Random Forest", "ORIGINAL", rf_original_pred_eval)

rf_enriched_features = enriched_features
rf_enriched_numeric = [col for col in rf_enriched_features if col not in categorical_features]
rf_enriched_model = Pipeline([
    ("preprocessor", build_preprocessor(rf_enriched_numeric, [col for col in categorical_features if col in rf_enriched_features], False)),
    ("model", RandomForestRegressor(n_estimators=250, min_samples_leaf=50, max_features=1.0, random_state=RANDOM_STATE, n_jobs=-1)),
])
rf_enriched_model.fit(train[rf_enriched_features], np.log1p(train[TARGET].to_numpy(dtype=float)))
rf_enriched_pred_eval = np.clip(np.expm1(rf_enriched_model.predict(eval_model[rf_enriched_features])), 0, None)
point_models[("Random Forest", "ENRICHED_SAFE_HISTORY")] = rf_enriched_model
point_predictions_eval[("Random Forest", "ENRICHED_SAFE_HISTORY")] = rf_enriched_pred_eval
append_eval_scores(point_rows, "Random Forest", "ENRICHED_SAFE_HISTORY", rf_enriched_pred_eval)

pd.DataFrame([row for row in point_rows if row["model"] == "Random Forest"])

# 17. Gradient Boosting

O Gradient Boosting usa perda Huber, `n_estimators=300`, `learning_rate=0.10`, `max_depth=7`, `min_samples_leaf=20`, `subsample=1.0` e `random_state=42`.

In [ ]:
gb_original_features = original_features
gb_original_numeric = [col for col in gb_original_features if col not in categorical_features]
gb_original_model = Pipeline([
    ("preprocessor", build_preprocessor(gb_original_numeric, [col for col in categorical_features if col in gb_original_features], False)),
    ("model", GradientBoostingRegressor(loss="huber", n_estimators=300, learning_rate=0.10, max_depth=7, min_samples_leaf=20, subsample=1.0, 
                                        random_state=RANDOM_STATE)),
])
gb_original_model.fit(train[gb_original_features], np.log1p(train[TARGET].to_numpy(dtype=float)))
gb_original_pred_eval = np.clip(np.expm1(gb_original_model.predict(eval_model[gb_original_features])), 0, None)
point_models[("Gradient Boosting", "ORIGINAL")] = gb_original_model
point_predictions_eval[("Gradient Boosting", "ORIGINAL")] = gb_original_pred_eval
append_eval_scores(point_rows, "Gradient Boosting", "ORIGINAL", gb_original_pred_eval)

gb_enriched_features = enriched_features
gb_enriched_numeric = [col for col in gb_enriched_features if col not in categorical_features]
gb_enriched_model = Pipeline([
    ("preprocessor", build_preprocessor(gb_enriched_numeric, [col for col in categorical_features if col in gb_enriched_features], False)),
    ("model", GradientBoostingRegressor(loss="huber", n_estimators=300, learning_rate=0.10, max_depth=7, min_samples_leaf=20, subsample=1.0, 
                                        random_state=RANDOM_STATE)),
])
gb_enriched_model.fit(train[gb_enriched_features], np.log1p(train[TARGET].to_numpy(dtype=float)))
gb_enriched_pred_eval = np.clip(np.expm1(gb_enriched_model.predict(eval_model[gb_enriched_features])), 0, None)
point_models[("Gradient Boosting", "ENRICHED_SAFE_HISTORY")] = gb_enriched_model
point_predictions_eval[("Gradient Boosting", "ENRICHED_SAFE_HISTORY")] = gb_enriched_pred_eval
append_eval_scores(point_rows, "Gradient Boosting", "ENRICHED_SAFE_HISTORY", gb_enriched_pred_eval)

pd.DataFrame([row for row in point_rows if row["model"] == "Gradient Boosting"])

# 18. HistGradientBoosting

O HistGradientBoosting usa tratamento categórico por dtype. As categorias são aprendidas no treino e reaplicadas na validação/calibração para evitar aprendizagem de categorias a partir do futuro.

In [ ]:
hgb_original_features = original_features
hgb_original_x_train, hgb_original_x_eval = prepare_hgb_frames(train, eval_model, hgb_original_features, categorical_features)
hgb_original_model = HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.10, max_iter=300, max_leaf_nodes=15, min_samples_leaf=20, 
                                                   l2_regularization=1.0, categorical_features="from_dtype", early_stopping=False, 
                                                   random_state=RANDOM_STATE)
hgb_original_model.fit(hgb_original_x_train, np.log1p(train[TARGET].to_numpy(dtype=float)))
hgb_original_pred_eval = np.clip(np.expm1(hgb_original_model.predict(hgb_original_x_eval)), 0, None)
point_models[("HistGradientBoosting", "ORIGINAL")] = hgb_original_model
point_predictions_eval[("HistGradientBoosting", "ORIGINAL")] = hgb_original_pred_eval
append_eval_scores(point_rows, "HistGradientBoosting", "ORIGINAL", hgb_original_pred_eval)

hgb_enriched_features = enriched_features
hgb_enriched_x_train, hgb_enriched_x_eval = prepare_hgb_frames(train, eval_model, hgb_enriched_features, categorical_features)
hgb_enriched_model = HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.10, max_iter=300, max_leaf_nodes=15, min_samples_leaf=20, 
                                                   l2_regularization=1.0, categorical_features="from_dtype", early_stopping=False, 
                                                   random_state=RANDOM_STATE)
hgb_enriched_model.fit(hgb_enriched_x_train, np.log1p(train[TARGET].to_numpy(dtype=float)))
hgb_enriched_pred_eval = np.clip(np.expm1(hgb_enriched_model.predict(hgb_enriched_x_eval)), 0, None)
point_models[("HistGradientBoosting", "ENRICHED_SAFE_HISTORY")] = hgb_enriched_model
point_predictions_eval[("HistGradientBoosting", "ENRICHED_SAFE_HISTORY")] = hgb_enriched_pred_eval
append_eval_scores(point_rows, "HistGradientBoosting", "ENRICHED_SAFE_HISTORY", hgb_enriched_pred_eval)

pd.DataFrame([row for row in point_rows if row["model"] == "HistGradientBoosting"])

# 19. Comparação dos modelos em validação

A comparação abaixo é construída diretamente das previsões recém-calculadas. A ordenação segue o critério oficial: MAE, depois RMSE, depois MAE na cauda Q90.

In [ ]:
point_comparison = pd.DataFrame(point_rows)
validation_baseline_for_comparison = baseline_validation.rename(columns={"model": "model"}).copy()
validation_comparison = pd.concat(
    [validation_baseline_for_comparison, point_comparison.query("dataset == 'validation'")],
    ignore_index=True,
).sort_values(["mae", "rmse", "mae_q90"]).reset_index(drop=True)
validation_comparison.to_csv(RUN_RESULTS / "validation_comparison.csv", index=False)
validation_comparison[["model", "feature_config", "dataset", "mae", "rmse", "medae", "rmsle", "mae_q90", "mae_q95"]]

# 20. Seleção programática do modelo vencedor

O vencedor emerge dos resultados em `validation`. O `final_test` ainda não foi usado.

In [ ]:
winner_candidates = point_comparison[point_comparison["dataset"] == "validation"].copy()
winner = winner_candidates.sort_values(["mae", "rmse", "mae_q90"]).reset_index(drop=True).iloc[0].to_dict()
winner["selection_criterion"] = "Menor MAE em validation; desempate por RMSE e MAE na cauda Q90. Calibration é reportado como verificação secundária, sem consulta ao final_test."
pd.DataFrame([winner])[["model", "feature_config", "dataset", "mae", "rmse", "mae_q90", "selection_criterion"]]

# 21. Avaliação em calibration

A calibração é usada como verificação secundária da especificação escolhida em validação. Ela não altera o modelo vencedor, as features ou os hiperparâmetros.

In [ ]:
calibration_comparison = point_comparison.query("dataset == 'calibration'").sort_values(["mae", "rmse", "mae_q90"]).reset_index(drop=True)
calibration_comparison.to_csv(RUN_RESULTS / "calibration_comparison.csv", index=False)
calibration_comparison[["model", "feature_config", "dataset", "mae", "rmse", "medae", "rmsle", "mae_q90", "mae_q95"]]

# 22. Congelamento da especificação final

A partir deste ponto, a especificação está congelada. O teste final pode ser aberto apenas para avaliação de generalização temporal.

In [ ]:
selected_spec = {
    "modelo_selecionado": winner["model"],
    "feature_config": winner["feature_config"],
    "hiperparametros": {
        "loss": "squared_error" if winner["model"] == "HistGradientBoosting" else "ver células anteriores",
        "learning_rate": 0.10,
        "max_iter": 300,
        "max_leaf_nodes": 15,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0,
        "random_state": RANDOM_STATE,
    },
    "selecao_encerrada": True,
}
pd.DataFrame([selected_spec])

# 23. Treinamento final

O treino final combina `train`, `validation` e `calibration`. A avaliação pareada treina novamente, do zero, o mesmo modelo vencedor com duas configurações: `ORIGINAL` e `ENRICHED_SAFE_HISTORY`. A única diferença entre os dois modelos é o conjunto de features.

In [ ]:
final_train = pd.concat([train, validation, calibration], ignore_index=True)
final_test = splits["final_test"].copy()
q90_final_train = float(final_train[TARGET].quantile(0.90))
q95_final_train = float(final_train[TARGET].quantile(0.95))

final_original_features = original_features
final_original_x_train, final_original_x_test = prepare_hgb_frames(final_train, final_test, final_original_features, categorical_features)
final_original_model = HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.10, max_iter=300, max_leaf_nodes=15, 
                                                     min_samples_leaf=20, l2_regularization=1.0, categorical_features="from_dtype", 
                                                     early_stopping=False, random_state=RANDOM_STATE)
final_original_model.fit(final_original_x_train, np.log1p(final_train[TARGET].to_numpy(dtype=float)))
final_original_pred = np.clip(np.expm1(final_original_model.predict(final_original_x_test)), 0, None)

final_enriched_features = enriched_features
final_enriched_x_train, final_enriched_x_test = prepare_hgb_frames(final_train, final_test, final_enriched_features, categorical_features)
final_enriched_model = HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.10, max_iter=300, max_leaf_nodes=15, 
                                                     min_samples_leaf=20, l2_regularization=1.0, categorical_features="from_dtype", 
                                                     early_stopping=False, random_state=RANDOM_STATE)
final_enriched_model.fit(final_enriched_x_train, np.log1p(final_train[TARGET].to_numpy(dtype=float)))
final_enriched_pred = np.clip(np.expm1(final_enriched_model.predict(final_enriched_x_test)), 0, None)
print("Modelos finais treinados do zero.")

# 24. Teste final

Somente agora o `final_test` é usado. Os ganhos são calculados programaticamente a partir das previsões recém-geradas.

In [ ]:
final_comparison = pd.DataFrame([
    score_row(str(winner["model"]), "ORIGINAL", "final_test", final_test[TARGET], final_original_pred, q90_final_train, q95_final_train),
    score_row(str(winner["model"]), "ENRICHED_SAFE_HISTORY", "final_test", final_test[TARGET], final_enriched_pred, q90_final_train, q95_final_train),
])
mae_original = float(final_comparison.loc[final_comparison["feature_config"] == "ORIGINAL", "mae"].iloc[0])
mae_enriched = float(final_comparison.loc[final_comparison["feature_config"] == "ENRICHED_SAFE_HISTORY", "mae"].iloc[0])
mae_gain_h = mae_original - mae_enriched
mae_gain_pct = mae_gain_h / mae_original * 100
final_comparison["mae_gain_h_vs_original"] = [0.0, mae_gain_h]
final_comparison["mae_gain_pct_vs_original"] = [0.0, mae_gain_pct]
final_comparison.to_csv(RUN_RESULTS / "final_comparison.csv", index=False)
final_comparison

In [ ]:
final_predictions = final_test[[col for col in ["port_call_id", PORT_COL, "port_display", OPERATION_COL, DATE_COL, TARGET] if col in final_test.columns]].copy()
final_predictions["selected_model"] = winner["model"]
final_predictions["prediction_original_h"] = final_original_pred
final_predictions["prediction_enriched_h"] = final_enriched_pred
final_predictions["abs_error_original_h"] = np.abs(final_original_pred - final_predictions[TARGET])
final_predictions["abs_error_enriched_h"] = np.abs(final_enriched_pred - final_predictions[TARGET])
final_predictions["error_original_h"] = final_original_pred - final_predictions[TARGET]
final_predictions["error_enriched_h"] = final_enriched_pred - final_predictions[TARGET]
final_predictions["arrival_week"] = final_predictions[DATE_COL].dt.to_period("W").astype(str)
final_predictions.to_parquet(RUN_RESULTS / "final_predictions.parquet", index=False)
final_predictions.head()

# 25. Análise da longa cauda

A cauda é avaliada no `final_test` a partir das previsões recém-geradas. Bias negativo e alta taxa de underprediction indicam subestimação de permanências longas.

In [ ]:
tail_comparison = final_comparison[[
    "model", "feature_config", "dataset", "mae_q90", "mae_q95", "bias_q90", "bias_q95", "underprediction_rate_q90", "underprediction_rate_q95",
]].copy()
tail_comparison.to_csv(RUN_RESULTS / "tail_comparison.csv", index=False)
tail_comparison

# 26. Bootstrap pareado

O bootstrap estima a incerteza do ganho de erro absoluto entre `ORIGINAL` e `ENRICHED_SAFE_HISTORY`. São usados três desenhos: pareado global, blocos por semana e blocos por porto × semana.

In [ ]:
def paired_bootstrap(df: pd.DataFrame, label: str, n_boot: int = 1000) -> dict[str, object]:
    diffs = df["abs_error_original_h"].to_numpy() - df["abs_error_enriched_h"].to_numpy()
    rng = np.random.default_rng(RANDOM_STATE)
    reps = np.empty(n_boot)
    for i in range(n_boot):
        reps[i] = rng.choice(diffs, size=len(diffs), replace=True).mean()
    return {"method": label, "n": int(len(df)), "mean_abs_error_gain_h": float(diffs.mean()), "ci95_low_h": float(np.quantile(reps, 0.025)), 
            "ci95_high_h": float(np.quantile(reps, 0.975)), "status": "ok"}


def block_bootstrap(df: pd.DataFrame, block_cols: list[str], label: str, n_boot: int = 800) -> dict[str, object]:
    temp = df.copy()
    temp["_block"] = temp[block_cols].astype("string").fillna("__MISSING__").agg("|".join, axis=1)
    blocks = temp["_block"].drop_duplicates().to_numpy()
    if len(blocks) < 10:
        return {"method": label, "n_blocks": int(len(blocks)), "mean_abs_error_gain_h": np.nan, "ci95_low_h": np.nan, "ci95_high_h": np.nan, 
                "status": "not_enough_blocks"}
    rng = np.random.default_rng(RANDOM_STATE)
    reps = np.empty(n_boot)
    for i in range(n_boot):
        sample_blocks = rng.choice(blocks, size=len(blocks), replace=True)
        sample = pd.concat([temp[temp["_block"] == block] for block in sample_blocks], ignore_index=True)
        reps[i] = sample["abs_error_original_h"].mean() - sample["abs_error_enriched_h"].mean()
    return {"method": label, "n_blocks": int(len(blocks)), 
            "mean_abs_error_gain_h": float(temp["abs_error_original_h"].mean() - temp["abs_error_enriched_h"].mean()), 
            "ci95_low_h": float(np.quantile(reps, 0.025)), "ci95_high_h": float(np.quantile(reps, 0.975)), "status": "ok"}

bootstrap_results = pd.DataFrame([
    paired_bootstrap(final_predictions, "paired_global"),
    block_bootstrap(final_predictions, ["arrival_week"], "block_week"),
    block_bootstrap(final_predictions, [PORT_COL, "arrival_week"], "block_port_week"),
])
bootstrap_results.to_csv(RUN_RESULTS / "bootstrap_results.csv", index=False)
bootstrap_results

# 27. Importância das features

A importância por permutação mede quanto o MAE aumenta quando uma feature é embaralhada. Como há correlação entre variáveis, as importâncias não devem ser somadas mecanicamente.

In [ ]:
def infer_feature_family(feature: str) -> str:
    if feature in {PORT_COL, "region", "state"}:
        return "ESTRUTURAL_GEOGRAFICA"
    if feature == OPERATION_COL or feature.startswith("op_"):
        return "OPERACIONAL_DA_ESCALA"
    if feature.startswith("arrival_"):
        return "TEMPORAL"
    if feature.endswith(("_prev_1d", "_prev_3d", "_prev_7d")):
        if feature.startswith(("rain_", "precipitation_", "wind_", "temperature_")):
            return "CLIMATICA_HISTORICA"
        return "FLUXO_PORTUARIO_HISTORICO"
    return "OUTRA"


def predict_original_scale(model: object, x: pd.DataFrame) -> np.ndarray:
    return np.clip(np.expm1(model.predict(x)), 0, None)


def permutation_importance_manual(model: object, x_test: pd.DataFrame, y_true: np.ndarray, features: list[str], 
                                  original_feature_set: set[str], feature_metadata: dict[str, str], 
                                  sample_size: int = 2500, n_repeats: int = 3) -> pd.DataFrame:
    rng = np.random.default_rng(RANDOM_STATE)
    if len(x_test) > sample_size:
        positions = rng.choice(len(x_test), size=sample_size, replace=False)
        x_sample = x_test.iloc[positions].copy()
        y_sample = y_true[positions]
    else:
        x_sample = x_test.copy()
        y_sample = y_true
    base_pred = predict_original_scale(model, x_sample)
    base_mae = mean_absolute_error(y_sample, base_pred)
    rows = []
    for feature in features:
        deltas = []
        for _ in range(n_repeats):
            shuffled = x_sample.copy()
            shuffled[feature] = rng.permutation(shuffled[feature].to_numpy())
            pred = predict_original_scale(model, shuffled)
            deltas.append(mean_absolute_error(y_sample, pred) - base_mae)
        rows.append({"feature": feature, "feature_origin": "original" if feature in original_feature_set else "new", 
                     "family": feature_metadata.get(feature, infer_feature_family(feature)), 
                     "importance_mae_increase_h": float(np.mean(deltas)), "importance_std_h": float(np.std(deltas)), 
                     "sample_size": int(len(x_sample))})
    return pd.DataFrame(rows).sort_values("importance_mae_increase_h", ascending=False)

feature_importance = permutation_importance_manual(final_enriched_model, final_enriched_x_test, final_test[TARGET].to_numpy(dtype=float), 
                                                   enriched_features, set(original_features), registry.feature_metadata)
feature_importance.to_csv(RUN_RESULTS / "feature_importance.csv", index=False)
feature_importance.head(20)

# 28. Ablation / valor incremental das famílias de features

A ablação reexecuta a família de árvore selecionada em conjuntos progressivos de variáveis. O objetivo é entender o valor incremental de estrutura, tempo, clima e histórico seguro.

In [ ]:
def historical_feature_sets(registry: FeatureRegistry) -> dict[str, list[str]]:
    base = registry.original_features
    fam = registry.historical_families
    sets = {
        "E0_original": base,
        "E1_fluxo_d1": base + ["arrivals_prev_1d", "departures_prev_1d", "flow_balance_prev_1d"],
        "E2_fluxo_1_3d": base + ["arrivals_prev_1d", "arrivals_prev_3d", "arrivals_avg_prev_3d", "arrivals_max_prev_3d", "departures_prev_1d", 
                                 "departures_prev_3d", "departures_avg_prev_3d", "flow_balance_prev_1d", "flow_balance_prev_3d"],
        "E3_fluxo_1_3_7d": base + ["arrivals_prev_1d", "arrivals_prev_3d", "arrivals_prev_7d", "arrivals_avg_prev_3d", "arrivals_avg_prev_7d", 
                                   "arrivals_max_prev_3d", "arrivals_max_prev_7d", "departures_prev_1d", "departures_prev_3d", 
                                   "departures_prev_7d", "departures_avg_prev_3d", "departures_avg_prev_7d", "flow_balance_prev_1d", 
                                   "flow_balance_prev_3d", "flow_balance_prev_7d"],
    }
    sets["E4_desempenho_porto"] = sets["E3_fluxo_1_3_7d"] + fam.get("port_performance", [])
    sets["E5_estado_operacional"] = sets["E4_desempenho_porto"] + fam.get("state", [])
    sets["E6_historico_adicional"] = registry.enriched_features
    return {name: list(dict.fromkeys([col for col in cols if col in df_model.columns])) for name, cols in sets.items()}


def contextual_feature_sets(registry: FeatureRegistry) -> dict[str, list[str]]:
    structure = ["port"] + registry.operation_features
    time_features = ["arrival_shift", "arrival_season", "arrival_quarter", "arrival_weekofyear", "arrival_is_weekend", "arrival_hour_sin", 
                     "arrival_hour_cos", "arrival_dow_sin", "arrival_dow_cos", "arrival_month_sin", "arrival_month_cos"]
    weather = [col for col in registry.numerical_features if infer_feature_family(col) == "CLIMATICA_HISTORICA"]
    port_hist = registry.historical_families.get("port_performance", [])
    state = registry.historical_families.get("state", [])
    route_vessel = registry.historical_families.get("route_history", []) + registry.historical_families.get("vessel_history", []) + registry.historical_families.get("vessel_port_history", []) + registry.historical_families.get("source_history", []) + registry.historical_families.get("destination_history", [])
    sets = {
        "1_estrutura": structure,
        "2_estrutura_tempo": structure + time_features,
        "3_estrutura_clima": structure + weather,
        "4_estrutura_tempo_clima": structure + time_features + weather,
        "5_historico_porto": structure + time_features + weather + port_hist,
        "6_estado_operacional_d1": structure + time_features + weather + port_hist + state,
        "7_historico_rota_navio": structure + time_features + weather + port_hist + state + route_vessel,
        "8_full_model": registry.enriched_features,
    }
    return {name: list(dict.fromkeys([col for col in cols if col in df_model.columns])) for name, cols in sets.items()}

In [ ]:
def fit_predict_hgb_for_features(train_df: pd.DataFrame, eval_df: pd.DataFrame, features: list[str]) -> np.ndarray:
    x_train, x_eval = prepare_hgb_frames(train_df, eval_df, features, [col for col in categorical_features if col in features])
    model = HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.10, max_iter=300, max_leaf_nodes=15, min_samples_leaf=20, 
                                          l2_regularization=1.0, categorical_features="from_dtype", early_stopping=False, random_state=RANDOM_STATE)
    model.fit(x_train, np.log1p(train_df[TARGET].to_numpy(dtype=float)))
    return np.clip(np.expm1(model.predict(x_eval)), 0, None)

ablation_rows = []
for set_name, features in historical_feature_sets(registry).items():
    pred_eval = fit_predict_hgb_for_features(train, eval_model, features)
    append_eval_scores(ablation_rows, set_name, "ABLAÇÃO_HISTORICA", pred_eval)

historical_ablation = pd.DataFrame(ablation_rows)
base_by_dataset = historical_ablation[historical_ablation["model"] == "E0_original"][["dataset", "mae"]].rename(columns={"mae": "e0_mae"})
historical_ablation = historical_ablation.merge(base_by_dataset, on="dataset", how="left")
historical_ablation["mae_gain_vs_e0_h"] = historical_ablation["e0_mae"] - historical_ablation["mae"]
historical_ablation["mae_gain_vs_e0_pct"] = historical_ablation["mae_gain_vs_e0_h"] / historical_ablation["e0_mae"] * 100
historical_ablation.to_csv(RUN_RESULTS / "historical_feature_ablation.csv", index=False)
historical_ablation.query("dataset == 'validation'")[["model", "mae", "mae_q90", "mae_gain_vs_e0_h", "mae_gain_vs_e0_pct"]]

In [ ]:
contextual_rows = []
for set_name, features in contextual_feature_sets(registry).items():
    pred_eval = fit_predict_hgb_for_features(train, eval_model, features)
    append_eval_scores(contextual_rows, set_name, "CONTEXTUAL", pred_eval)

contextual_ablation = pd.DataFrame(contextual_rows)
contextual_ablation.to_csv(RUN_RESULTS / "contextual_ablation.csv", index=False)
contextual_ablation.query("dataset == 'validation'")[["model", "mae", "mae_q90"]]

# 29. Regressão quantílica

Os modelos quantílicos estimam P50, P90 e P95 usando HistGradientBoosting com `loss='quantile'` e target `log1p`. Eles são treinados após o fluxo pontual estar congelado.

In [ ]:
quantile_train = pd.concat([train, validation], ignore_index=True)
quantile_eval = pd.concat([calibration, final_test], ignore_index=True)
n_cal = len(calibration)
QUANTILES = {"p50": 0.50, "p90": 0.90, "p95": 0.95}

quantile_models_original = {}
quantile_raw_original = {}
q_original_x_train, q_original_x_eval = prepare_hgb_frames(quantile_train, quantile_eval, original_features, categorical_features)
for q_name, q_alpha in QUANTILES.items():
    quantile_models_original[q_name] = HistGradientBoostingRegressor(loss="quantile", quantile=q_alpha, learning_rate=0.10, max_iter=300, 
                                                                     max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0, 
                                                                     categorical_features="from_dtype", early_stopping=False, random_state=RANDOM_STATE)
    quantile_models_original[q_name].fit(q_original_x_train, np.log1p(quantile_train[TARGET].to_numpy(dtype=float)))
    quantile_raw_original[q_name] = np.clip(np.expm1(quantile_models_original[q_name].predict(q_original_x_eval)), 0, None)

quantile_models_enriched = {}
quantile_raw_enriched = {}
q_enriched_x_train, q_enriched_x_eval = prepare_hgb_frames(quantile_train, quantile_eval, enriched_features, categorical_features)
for q_name, q_alpha in QUANTILES.items():
    quantile_models_enriched[q_name] = HistGradientBoostingRegressor(loss="quantile", quantile=q_alpha, learning_rate=0.10, max_iter=300, 
                                                                     max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0, 
                                                                     categorical_features="from_dtype", early_stopping=False, random_state=RANDOM_STATE)
    quantile_models_enriched[q_name].fit(q_enriched_x_train, np.log1p(quantile_train[TARGET].to_numpy(dtype=float)))
    quantile_raw_enriched[q_name] = np.clip(np.expm1(quantile_models_enriched[q_name].predict(q_enriched_x_eval)), 0, None)

print("Modelos quantílicos P50/P90/P95 treinados para ORIGINAL e ENRICHED_SAFE_HISTORY.")

# 30. Rearranjo e crossing de quantis

Antes do rearranjo, pode ocorrer crossing, como P90 menor que P50. A regra monotônica aplica `maximum.accumulate` para garantir `P50 <= P90 <= P95`.

In [ ]:
def crossing_rate(preds: dict[str, np.ndarray]) -> float:
    return float(np.mean((preds["p50"] > preds["p90"]) | (preds["p90"] > preds["p95"])))


def rearrange_quantiles(preds: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    stacked = np.vstack([preds["p50"], preds["p90"], preds["p95"]])
    ordered = np.maximum.accumulate(stacked, axis=0)
    return {"p50": ordered[0], "p90": ordered[1], "p95": ordered[2]}

crossing_raw = {
    "ORIGINAL": crossing_rate(quantile_raw_original),
    "ENRICHED_SAFE_HISTORY": crossing_rate(quantile_raw_enriched),
}
quantile_preds_original = rearrange_quantiles(quantile_raw_original)
quantile_preds_enriched = rearrange_quantiles(quantile_raw_enriched)
pd.DataFrame([{"feature_config": k, "quantile_crossing_rate_raw": v} for k, v in crossing_raw.items()])

# 31. Avaliação quantílica

A avaliação usa pinball loss, cobertura empírica e largura média dos intervalos P90-P50 e P95-P50. Também é reconstruído o baseline quantílico hierárquico.

In [ ]:
quantile_rows = []
coverage_rows = []
prediction_tables = []
for feature_config, preds in [("ORIGINAL", quantile_preds_original), ("ENRICHED_SAFE_HISTORY", quantile_preds_enriched)]:
    for dataset, part, start, stop in [("calibration", calibration, 0, n_cal), ("final_test", final_test, n_cal, len(quantile_eval))]:
        y = part[TARGET].to_numpy(dtype=float)
        row = {"model": "hgb_quantile_log", "feature_config": feature_config, "dataset": dataset, 
               "quantile_crossing_rate_raw": crossing_raw[feature_config], 
               "mean_width_p90_p50_h": float(np.mean(preds["p90"][start:stop] - preds["p50"][start:stop])), 
               "mean_width_p95_p50_h": float(np.mean(preds["p95"][start:stop] - preds["p50"][start:stop]))}
        pred_table = part[[col for col in ["port_call_id", PORT_COL, OPERATION_COL, DATE_COL, TARGET] if col in part.columns]].copy()
        pred_table["feature_config"] = feature_config
        pred_table["dataset"] = dataset
        for q_name, q_alpha in QUANTILES.items():
            pred = preds[q_name][start:stop]
            row[f"pinball_{q_name}"] = float(mean_pinball_loss(y, pred, alpha=q_alpha))
            coverage_rows.append({"model": "hgb_quantile_log", "feature_config": feature_config, "dataset": dataset, 
                                  "quantile": q_name, "target_coverage": q_alpha, "empirical_coverage": float(np.mean(y <= pred)), 
                                  "pinball_loss": row[f"pinball_{q_name}"]})
            pred_table[f"pred_{q_name}_h"] = pred
        pred_table["width_p90_p50_h"] = pred_table["pred_p90_h"] - pred_table["pred_p50_h"]
        pred_table["width_p95_p50_h"] = pred_table["pred_p95_h"] - pred_table["pred_p50_h"]
        prediction_tables.append(pred_table)
        quantile_rows.append(row)

for dataset, part in [("calibration", calibration), ("final_test", final_test)]:
    preds_base = {q_name: hierarchical_quantile_prediction(quantile_train, part, q_alpha) for q_name, q_alpha in QUANTILES.items()}
    preds_base = rearrange_quantiles(preds_base)
    y = part[TARGET].to_numpy(dtype=float)
    row = {"model": "baseline_quantile_hierarchical", "feature_config": "ORIGINAL", "dataset": dataset, 
           "quantile_crossing_rate_raw": 0.0, "mean_width_p90_p50_h": float(np.mean(preds_base["p90"] - preds_base["p50"])), 
           "mean_width_p95_p50_h": float(np.mean(preds_base["p95"] - preds_base["p50"]))}
    for q_name, q_alpha in QUANTILES.items():
        row[f"pinball_{q_name}"] = float(mean_pinball_loss(y, preds_base[q_name], alpha=q_alpha))
        coverage_rows.append({"model": "baseline_quantile_hierarchical", "feature_config": "ORIGINAL", "dataset": dataset, 
                              "quantile": q_name, "target_coverage": q_alpha, "empirical_coverage": float(np.mean(y <= preds_base[q_name])), 
                              "pinball_loss": row[f"pinball_{q_name}"]})
    quantile_rows.append(row)

quantile_comparison = pd.DataFrame(quantile_rows)
quantile_coverage = pd.DataFrame(coverage_rows)
quantile_predictions = pd.concat(prediction_tables, ignore_index=True)
quantile_comparison.to_csv(RUN_RESULTS / "quantile_comparison.csv", index=False)
quantile_coverage.to_csv(RUN_RESULTS / "quantile_coverage.csv", index=False)
quantile_predictions.to_parquet(RUN_RESULTS / "quantile_predictions.parquet", index=False)
quantile_comparison

# 32. Análise da incerteza

A largura P95-P50 é tratada como indicador de incerteza. A análise observa associação com erro absoluto, permanência real e quintis de risco, além de uma leitura intragrupo porto × operação.

In [ ]:
def safe_spearman(a: pd.Series, b: pd.Series) -> float:
    valid = pd.DataFrame({"a": a, "b": b}).dropna()
    if len(valid) < 5 or valid["a"].nunique() < 2 or valid["b"].nunique() < 2:
        return np.nan
    return float(spearmanr(valid["a"], valid["b"]).correlation)

uncertainty_df = quantile_predictions[(quantile_predictions["dataset"] == "final_test") & (quantile_predictions["feature_config"] == "ENRICHED_SAFE_HISTORY")].copy()
uncertainty_df["abs_error_p50_h"] = np.abs(uncertainty_df["pred_p50_h"] - uncertainty_df[TARGET])
uncertainty_df["extreme_q90"] = uncertainty_df[TARGET] >= uncertainty_df[TARGET].quantile(0.90)

global_rows = []
for width_col in ["width_p90_p50_h", "width_p95_p50_h"]:
    global_rows.append({"measure": width_col, "spearman_abs_error": safe_spearman(uncertainty_df[width_col], uncertainty_df["abs_error_p50_h"]), 
                        "spearman_actual_stay": safe_spearman(uncertainty_df[width_col], uncertainty_df[TARGET]), 
                        "mean_width_h": float(uncertainty_df[width_col].mean()), "median_width_h": float(uncertainty_df[width_col].median())})
uncertainty_df["risk_quintile"] = pd.qcut(uncertainty_df["width_p95_p50_h"].rank(method="first"), 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
quintiles = uncertainty_df.groupby("risk_quintile", observed=True).agg(n=("port_call_id", "count"), mean_abs_error_h=("abs_error_p50_h", "mean"), 
                                                                       mean_actual_h=(TARGET, "mean"), extreme_rate=("extreme_q90", "mean"), 
                                                                       mean_width_p95_p50_h=("width_p95_p50_h", "mean")).reset_index()
quintiles.insert(0, "measure", "risk_quintile")
uncertainty_global = pd.concat([pd.DataFrame(global_rows), quintiles], ignore_index=True)
uncertainty_global.to_csv(RUN_RESULTS / "uncertainty_global.csv", index=False)
uncertainty_global

In [ ]:
within_rows = []
for keys, group in uncertainty_df.groupby([PORT_COL, OPERATION_COL], dropna=False):
    if len(group) < MIN_GROUP_SIZE:
        continue
    within_rows.append({PORT_COL: keys[0], OPERATION_COL: keys[1], "n": int(len(group)), 
                        "spearman_width_abs_error": safe_spearman(group["width_p95_p50_h"], group["abs_error_p50_h"]), 
                        "spearman_width_actual": safe_spearman(group["width_p95_p50_h"], group[TARGET]), 
                        "hist_actual_std_h": float(group[TARGET].std()), "mean_width_p95_p50_h": float(group["width_p95_p50_h"].mean())})
uncertainty_within_group = pd.DataFrame(within_rows)
if not uncertainty_within_group.empty:
    uncertainty_within_group["can_individualize_risk"] = uncertainty_within_group["spearman_width_abs_error"] > 0.20
uncertainty_within_group.to_csv(RUN_RESULTS / "uncertainty_within_group.csv", index=False)
uncertainty_within_group.head(20)

# 33. Simulação de estoque de segurança

A simulação usa somente previsões produzidas nesta execução do notebook. Ela é uma análise de cenário: o dataset não contém demanda real da empresa, custo unitário real, estoque real ou economia financeira observada.

# 34. CORREÇÃO metodológica obrigatória na política estática

A política `A_static_segment_port_operation` é calculada com P90 histórico de `porto × tipo de operação` usando apenas dados anteriores ao `final_test`: `train + validation + calibration`. Assim, o benchmark estático não usa o target realizado do próprio período avaliado.

In [ ]:
stock_df = final_predictions.copy()
q_final = quantile_predictions[(quantile_predictions["dataset"] == "final_test") & (quantile_predictions["feature_config"] == "ENRICHED_SAFE_HISTORY")][["port_call_id", "pred_p50_h", "pred_p90_h", "pred_p95_h"]]
stock_df = stock_df.merge(q_final, on="port_call_id", how="left")
stock_df["actual_lead_time_d"] = stock_df[TARGET] / 24
stock_df["original_lead_time_d"] = stock_df["prediction_original_h"] / 24
stock_df["enriched_lead_time_d"] = stock_df["prediction_enriched_h"] / 24
stock_df["quantile_p90_lead_time_d"] = stock_df["pred_p90_h"] / 24
stock_df["quantile_p95_lead_time_d"] = stock_df["pred_p95_h"] / 24

historical_train = final_train.copy()
historical_train["historical_lead_time_d"] = historical_train[TARGET] / 24
segment_p90 = historical_train.groupby([PORT_COL, OPERATION_COL], dropna=False)["historical_lead_time_d"].quantile(0.90).rename("static_segment_p90_d").reset_index()
port_p90 = historical_train.groupby([PORT_COL], dropna=False)["historical_lead_time_d"].quantile(0.90).rename("static_port_p90_d").reset_index()
global_p90 = float(historical_train["historical_lead_time_d"].quantile(0.90))
stock_df = stock_df.merge(segment_p90, on=[PORT_COL, OPERATION_COL], how="left").merge(port_p90, on=[PORT_COL], how="left")
stock_df["static_segment_p90_d"] = stock_df["static_segment_p90_d"].fillna(stock_df["static_port_p90_d"]).fillna(global_p90)
stock_df[[PORT_COL, OPERATION_COL, "actual_lead_time_d", "static_segment_p90_d"]].head()

# 35. Políticas da simulação

As políticas comparam baseline estático histórico, modelos pontuais e quantis P90/P95. As previsões vêm da execução atual do notebook.

In [ ]:
policies = {
    "A_static_segment_port_operation": "static_segment_p90_d",
    "B_dynamic_original_point": "original_lead_time_d",
    "C_dynamic_enriched_point": "enriched_lead_time_d",
    "D_dynamic_quantile_p90": "quantile_p90_lead_time_d",
    "D_dynamic_quantile_p95": "quantile_p95_lead_time_d",
}
pd.DataFrame([{"policy": key, "lead_time_column": value} for key, value in policies.items()])

# 36. Cenários de demanda

Os cenários são hipóteses explícitas de simulação. Eles não representam demanda real observada.

In [ ]:
scenarios = pd.DataFrame([
    {"scenario": "base", "demand_mean_units_d": 100.0, "demand_std_units_d": 25.0, "service_z": 1.65, "unit_value_brl": 50.0},
    {"scenario": "low_demand", "demand_mean_units_d": 50.0, "demand_std_units_d": 15.0, "service_z": 1.65, "unit_value_brl": 50.0},
    {"scenario": "high_demand", "demand_mean_units_d": 200.0, "demand_std_units_d": 50.0, "service_z": 1.65, "unit_value_brl": 50.0},
    {"scenario": "higher_service", "demand_mean_units_d": 100.0, "demand_std_units_d": 25.0, "service_z": 2.05, "unit_value_brl": 50.0},
])
scenarios

# 37. Métricas da simulação

Para cada política são calculados lead time médio utilizado, estoque de segurança médio, proxy de capital de giro, protection rate, underbuffer rate e buffer médio excedente. Reduzir estoque só é positivo se o nível de proteção continuar aceitável para o cenário.

In [ ]:
stock_rows = []
for scenario in scenarios.to_dict("records"):
    mu = scenario["demand_mean_units_d"]
    sigma = scenario["demand_std_units_d"]
    z = scenario["service_z"]
    value = scenario["unit_value_brl"]
    for policy, col in policies.items():
        lead = stock_df[col].clip(lower=0).fillna(stock_df["static_segment_p90_d"])
        variance = sigma**2 * lead + mu**2 * lead.var()
        safety_stock = z * np.sqrt(np.maximum(variance, 0))
        buffer_excess_d = lead - stock_df["actual_lead_time_d"]
        stock_rows.append({**scenario, "policy": policy, "mean_policy_lead_time_d": float(lead.mean()), 
                           "mean_safety_stock_units": float(np.mean(safety_stock)), 
                           "proxy_working_capital_brl": float(np.mean(safety_stock) * value), 
                           "protection_rate": float((lead >= stock_df["actual_lead_time_d"]).mean()), 
                           "mean_buffer_excess_d": float(buffer_excess_d.mean()), "underbuffer_rate": float((buffer_excess_d < 0).mean())})

safety_stock_simulation = pd.DataFrame(stock_rows)
safety_stock_simulation.to_csv(RUN_RESULTS / "safety_stock_simulation.csv", index=False)
safety_stock_simulation.to_csv(RUN_RESULTS / "safety_stock_sensitivity.csv", index=False)
safety_stock_simulation.query("scenario == 'base'")

# 38. Síntese final do notebook

A síntese abaixo usa apenas variáveis calculadas durante a execução do notebook. Ela resume predição pontual, cauda, features históricas, quantis e simulação de estoque.

In [ ]:
quantile_width_final = float(
    quantile_comparison.query("feature_config == 'ENRICHED_SAFE_HISTORY' and dataset == 'final_test'")["mean_width_p95_p50_h"].iloc[0]
)
synthesis = pd.DataFrame(
    [
        {"bloco": "predição pontual", "resultado": f"{winner['model']} + {winner['feature_config']} selecionado em validation; ganho final de MAE = {mae_gain_h:.3f} h ({mae_gain_pct:.2f}%)."},
        {"bloco": "cauda", "resultado": f"MAE Q95 enriched = {tail_comparison.loc[tail_comparison['feature_config']=='ENRICHED_SAFE_HISTORY', 'mae_q95'].iloc[0]:.3f} h; eventos extremos continuam difíceis."},
        {"bloco": "features históricas", "resultado": f"{len(enriched_features) - len(original_features)} colunas históricas adicionais reconstruídas com cutoff temporal seguro."},
        {"bloco": "regressão quantílica", "resultado": f"P95-P50 médio no final_test enriched = {quantile_width_final:.3f} h."},
        {"bloco": "estoque", "resultado": "A simulação evidencia trade-off entre proteção e capital de giro proxy, sem afirmar economia real observada."},
    ]
)
synthesis

# 39. Verificação opcional contra resultados canônicos

Somente nesta seção os artefatos de `results/cap4_rebuild/` podem ser lidos. Eles servem apenas para comparar o resultado recém-calculado pelo notebook contra a execução canônica.

In [ ]:
OFFICIAL_RESULTS = ROOT / "results" / "cap4_rebuild"
verification_rows = []
if (OFFICIAL_RESULTS / "point_model_comparison_final.csv").exists():
    official_final = pd.read_csv(OFFICIAL_RESULTS / "point_model_comparison_final.csv")
    for config in ["ORIGINAL", "ENRICHED_SAFE_HISTORY"]:
        notebook_mae = float(final_comparison.loc[final_comparison["feature_config"] == config, "mae"].iloc[0])
        official_mae = float(official_final.loc[official_final["feature_config"] == config, "mae"].iloc[0])
        verification_rows.append({"metric": f"final_mae_{config}", "notebook": notebook_mae, "official": official_mae, "delta": notebook_mae - official_mae})
if (OFFICIAL_RESULTS / "quantile_model_comparison.csv").exists():
    official_quantile = pd.read_csv(OFFICIAL_RESULTS / "quantile_model_comparison.csv")
    nb_q = quantile_comparison.query("model == 'hgb_quantile_log' and feature_config == 'ENRICHED_SAFE_HISTORY' and dataset == 'final_test'").iloc[0]
    off_q = official_quantile.query("model == 'hgb_quantile_log' and feature_config == 'ENRICHED_SAFE_HISTORY' and dataset == 'final_test'").iloc[0]
    for metric in ["pinball_p50", "pinball_p90", "pinball_p95", "mean_width_p95_p50_h"]:
        verification_rows.append({"metric": f"quantile_{metric}", "notebook": float(nb_q[metric]), "official": float(off_q[metric]), "delta": float(nb_q[metric] - off_q[metric])})
reproducibility_check = pd.DataFrame(verification_rows)
reproducibility_check.to_csv(RUN_RESULTS / "reproducibility_check.csv", index=False)
reproducibility_check

# Geração das figuras finais do Capítulo 4

Todos os experimentos, resultados pontuais, regressão quantílica e simulação de estoque já foram concluídos nas seções anteriores. Esta seção apenas transforma os resultados calculados pelo próprio notebook em figuras finais para a monografia. Nenhuma nova modelagem é executada nesta etapa.


## Configuração visual e diretório de saída

A configuração abaixo replica o padrão gráfico usado no notebook final do Capítulo 3: Matplotlib, fundo branco, fontes discretas, bordas superior/direita removidas, grid leve, vírgula decimal e salvamento em PDF vetorial e PNG 300 dpi.


In [ ]:
import shutil
from matplotlib.ticker import FuncFormatter

CAP4_OUTPUT_DIR = ROOT / "outputs" / "capitulo4"
CAP4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MONOGRAFIA_IMG_DIR = ROOT.parent / "mba-tcc-monografia" / "USPSC-img"

print(f"Figuras do Capítulo 4 serão salvas em: {CAP4_OUTPUT_DIR.resolve()}")

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

COLOR_TOTAL = "#2f5d7c"
COLOR_MEDIAN = "#3b6f58"
COLOR_P90 = "#b06f2a"
COLOR_GRID = "#d9dde2"
COLOR_BOX = "#8fb3c8"
COLOR_MAP = "#4c78a8"


def fmt_decimal_pt(value, decimals=2):
    """Formata números com vírgula decimal para leitura das tabelas do capítulo."""
    if pd.isna(value):
        return ""
    return f"{value:,.{decimals}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_brl_pt(value, decimals=2):
    """Formata valores monetários em reais no padrão brasileiro."""
    return f"R$ {fmt_decimal_pt(value, decimals)}"


def relpath(path):
    """Mostra caminhos relativos ao repositório quando possível."""
    path = Path(path)
    try:
        return path.relative_to(ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def save_figure(fig, stem):
    """Salva uma figura em PDF vetorial e PNG 300 dpi no diretório do Capítulo 4."""
    pdf_path = CAP4_OUTPUT_DIR / f"{stem}.pdf"
    png_path = CAP4_OUTPUT_DIR / f"{stem}.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    return pdf_path, png_path


## Dados finais usados nas figuras

Os três blocos abaixo usam apenas objetos já calculados anteriormente no notebook: `contextual_ablation`, `uncertainty_global` e `safety_stock_simulation`. Os valores de referência são exibidos para detectar regressões, mas não são usados como fonte primária das figuras.


In [ ]:
ablation_label_map = {
    "1_estrutura": "Estrutura",
    "2_estrutura_tempo": "Estrutura + tempo",
    "3_estrutura_clima": "Estrutura + clima",
    "4_estrutura_tempo_clima": "Estrutura + tempo + clima",
    "5_historico_porto": "Histórico do porto",
    "6_estado_operacional_d1": "Estado operacional D-1",
    "7_historico_rota_navio": "Histórico de rota e embarcação",
    "8_full_model": "Modelo completo",
}
ablation_order = list(ablation_label_map)

cap4_ablation_plot = (
    contextual_ablation
    .query("dataset == 'validation'")
    .copy()
)
cap4_ablation_plot["label"] = cap4_ablation_plot["model"].map(ablation_label_map)
cap4_ablation_plot = cap4_ablation_plot.dropna(subset=["label"])
cap4_ablation_plot["ordem"] = cap4_ablation_plot["model"].map({name: i for i, name in enumerate(ablation_order)})
cap4_ablation_plot = cap4_ablation_plot.sort_values("ordem").reset_index(drop=True)
cap4_ablation_plot["ganho_mae_h"] = cap4_ablation_plot["mae"].iloc[0] - cap4_ablation_plot["mae"]
cap4_ablation_plot["ganho_mae_pct"] = cap4_ablation_plot["ganho_mae_h"] / cap4_ablation_plot["mae"].iloc[0] * 100

cap4_uncertainty_plot = (
    uncertainty_global
    .loc[uncertainty_global["measure"].eq("risk_quintile")]
    .copy()
    .sort_values("risk_quintile")
    .reset_index(drop=True)
)

policy_label_map = {
    "A_static_segment_port_operation": "P90 histórico",
    "B_dynamic_original_point": "Pontual inicial",
    "C_dynamic_enriched_point": "Pontual enriquecido",
    "D_dynamic_quantile_p90": "P90 dinâmico",
    "D_dynamic_quantile_p95": "P95 dinâmico",
}
policy_order = list(policy_label_map)
cap4_stock_plot = safety_stock_simulation.query("scenario == 'base'").copy()
cap4_stock_plot["policy_label"] = cap4_stock_plot["policy"].map(policy_label_map)
cap4_stock_plot = cap4_stock_plot.dropna(subset=["policy_label"])
cap4_stock_plot["ordem"] = cap4_stock_plot["policy"].map({name: i for i, name in enumerate(policy_order)})
cap4_stock_plot = cap4_stock_plot.sort_values("ordem").reset_index(drop=True)
cap4_stock_plot["coverage_pct"] = cap4_stock_plot["protection_rate"] * 100

consistency_check = pd.DataFrame([
    {
        "bloco": "ablação",
        "métrica": "MAE HGB inicial",
        "observado": float(cap4_ablation_plot.loc[cap4_ablation_plot["model"] == "2_estrutura_tempo", "mae"].iloc[0]),
        "referência": 47.30,
    },
    {
        "bloco": "ablação",
        "métrica": "MAE HGB histórico completo",
        "observado": float(cap4_ablation_plot.loc[cap4_ablation_plot["model"] == "8_full_model", "mae"].iloc[0]),
        "referência": 44.53,
    },
    {
        "bloco": "incerteza",
        "métrica": "Q1 erro absoluto médio P50",
        "observado": float(cap4_uncertainty_plot.loc[cap4_uncertainty_plot["risk_quintile"] == "Q1", "mean_abs_error_h"].iloc[0]),
        "referência": 8.7,
    },
    {
        "bloco": "incerteza",
        "métrica": "Q5 permanência média observada",
        "observado": float(cap4_uncertainty_plot.loc[cap4_uncertainty_plot["risk_quintile"] == "Q5", "mean_actual_h"].iloc[0]),
        "referência": 134.8,
    },
    {
        "bloco": "simulação",
        "métrica": "P90 histórico capital",
        "observado": float(cap4_stock_plot.loc[cap4_stock_plot["policy_label"] == "P90 histórico", "proxy_working_capital_brl"].iloc[0]),
        "referência": 37988.66,
    },
    {
        "bloco": "simulação",
        "métrica": "P95 dinâmico cobertura (%)",
        "observado": float(cap4_stock_plot.loc[cap4_stock_plot["policy_label"] == "P95 dinâmico", "coverage_pct"].iloc[0]),
        "referência": 94.41,
    },
])
consistency_check["diferença"] = consistency_check["observado"] - consistency_check["referência"]
consistency_check


## Figura 1 - Ablação das informações históricas

Mostra o ganho de MAE dos grupos de informação em relação à configuração inicial de referência, usando os resultados de validação já calculados na análise contextual.


In [ ]:
plot_df = cap4_ablation_plot.iloc[::-1].copy()

fig, ax = plt.subplots(figsize=(7.2, 4.6))
colors = np.where(plot_df["ganho_mae_h"] >= 0, COLOR_TOTAL, "#9b5f4a")
ax.barh(plot_df["label"], plot_df["ganho_mae_pct"], color=colors, alpha=0.9)
ax.axvline(0, color="#5c656b", linewidth=0.8)
ax.set_xlabel("Ganho de MAE em relação à configuração inicial (%)")
ax.grid(axis="x", color=COLOR_GRID, linewidth=0.7, alpha=0.9)
ax.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: fmt_decimal_pt(value, 1) + "%"))

for _, row in plot_df.iterrows():
    x = row["ganho_mae_pct"]
    ha = "left" if x >= 0 else "right"
    offset = 0.12 if x >= 0 else -0.12
    ax.text(
        x + offset,
        row["label"],
        f"{fmt_decimal_pt(row['mae'], 2)} h",
        va="center",
        ha=ha,
        fontsize=8,
    )

fig.tight_layout()
fig_ablacao_pdf, fig_ablacao_png = save_figure(fig, "cap4_ablacao_familias_historicas")

pd.DataFrame([{
    "figura": "cap4_ablacao_familias_historicas",
    "pdf": relpath(fig_ablacao_pdf),
    "png": relpath(fig_ablacao_png),
}])


## Figura 2 - Quintis de incerteza

Compara, por quintil da amplitude P95-P50, o erro absoluto médio do P50 e a permanência média observada.


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.2))
ax.plot(
    cap4_uncertainty_plot["risk_quintile"],
    cap4_uncertainty_plot["mean_abs_error_h"],
    marker="o",
    linewidth=2,
    color=COLOR_TOTAL,
    label="Erro absoluto médio do P50",
)
ax.plot(
    cap4_uncertainty_plot["risk_quintile"],
    cap4_uncertainty_plot["mean_actual_h"],
    marker="s",
    linewidth=2,
    color=COLOR_P90,
    label="Permanência média observada",
)
ax.set_xlabel("Quintil da amplitude P95-P50")
ax.set_ylabel("Horas")
ax.grid(axis="y", color=COLOR_GRID, linewidth=0.7, alpha=0.9)
ax.legend(frameon=False, loc="upper left")
y_max = max(cap4_uncertainty_plot["mean_abs_error_h"].max(), cap4_uncertainty_plot["mean_actual_h"].max())
ax.set_ylim(0, y_max * 1.12)
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, pos: fmt_decimal_pt(value, 0)))

for _, row in cap4_uncertainty_plot.iterrows():
    ax.annotate(
        fmt_decimal_pt(row["mean_abs_error_h"], 1),
        (row["risk_quintile"], row["mean_abs_error_h"]),
        xytext=(0, -14),
        textcoords="offset points",
        ha="center",
        fontsize=8,
        color=COLOR_TOTAL,
    )
    ax.annotate(
        fmt_decimal_pt(row["mean_actual_h"], 1),
        (row["risk_quintile"], row["mean_actual_h"]),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center",
        fontsize=8,
        color=COLOR_P90,
    )

fig.tight_layout()
fig_quintis_pdf, fig_quintis_png = save_figure(fig, "cap4_quintis_incerteza")

pd.DataFrame([{
    "figura": "cap4_quintis_incerteza",
    "pdf": relpath(fig_quintis_pdf),
    "png": relpath(fig_quintis_png),
}])


## Figura 3 - Trade-off capital × cobertura

Mostra o capital associado ao estoque de segurança e a taxa de cobertura do lead time para as cinco políticas avaliadas no cenário base da simulação.


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))
scatter = ax.scatter(
    cap4_stock_plot["proxy_working_capital_brl"],
    cap4_stock_plot["coverage_pct"],
    s=70,
    color=[COLOR_MEDIAN, "#6f8796", COLOR_TOTAL, COLOR_P90, "#9b5f4a"],
    edgecolor="white",
    linewidth=0.7,
    zorder=3,
)

label_offsets = {
    "P90 histórico": (8, 4, "left"),
    "Pontual inicial": (8, -8, "left"),
    "Pontual enriquecido": (8, 8, "left"),
    "P90 dinâmico": (-8, -12, "right"),
    "P95 dinâmico": (-8, 4, "right"),
}

for _, row in cap4_stock_plot.iterrows():
    dx, dy, ha = label_offsets.get(row["policy_label"], (6, 6, "left"))
    ax.annotate(
        row["policy_label"],
        (row["proxy_working_capital_brl"], row["coverage_pct"]),
        xytext=(dx, dy),
        textcoords="offset points",
        ha=ha,
        va="center",
        fontsize=8,
    )

ax.set_xlabel("Capital associado ao estoque de segurança")
ax.set_ylabel("Taxa de cobertura do lead time (%)")
ax.grid(color=COLOR_GRID, linewidth=0.7, alpha=0.9)
ax.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: fmt_brl_pt(value, 0)))
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, pos: fmt_decimal_pt(value, 0) + "%"))
ax.set_ylim(max(0, cap4_stock_plot["coverage_pct"].min() - 8), min(100, cap4_stock_plot["coverage_pct"].max() + 4))

fig.tight_layout()
fig_tradeoff_pdf, fig_tradeoff_png = save_figure(fig, "cap4_tradeoff_capital_cobertura")

pd.DataFrame([{
    "figura": "cap4_tradeoff_capital_cobertura",
    "pdf": relpath(fig_tradeoff_pdf),
    "png": relpath(fig_tradeoff_png),
}])


## Resumo dos outputs

A tabela final lista somente as três figuras finais do Capítulo 4. Quando o repositório da monografia existe como diretório irmão, apenas os PDFs finais são copiados para `../mba-tcc-monografia/USPSC-img/`.


In [ ]:
cap4_figure_outputs = pd.DataFrame([
    {
        "figura": "cap4_ablacao_familias_historicas",
        "objetivo": "Mostrar o ganho no MAE produzido pelos grupos de informações em relação à configuração inicial.",
        "caminho_pdf": relpath(fig_ablacao_pdf),
        "caminho_png": relpath(fig_ablacao_png),
    },
    {
        "figura": "cap4_quintis_incerteza",
        "objetivo": "Mostrar a relação entre amplitude P95-P50, erro absoluto médio do P50 e permanência média observada.",
        "caminho_pdf": relpath(fig_quintis_pdf),
        "caminho_png": relpath(fig_quintis_png),
    },
    {
        "figura": "cap4_tradeoff_capital_cobertura",
        "objetivo": "Mostrar o trade-off entre capital associado ao estoque de segurança e taxa de cobertura do lead time.",
        "caminho_pdf": relpath(fig_tradeoff_pdf),
        "caminho_png": relpath(fig_tradeoff_png),
    },
])

cap4_figure_outputs["existe_pdf"] = cap4_figure_outputs["caminho_pdf"].map(lambda p: (ROOT / p).exists())
cap4_figure_outputs["existe_png"] = cap4_figure_outputs["caminho_png"].map(lambda p: (ROOT / p).exists())

copied_pdf_names = set()
if MONOGRAFIA_IMG_DIR.exists():
    for pdf_path in [fig_ablacao_pdf, fig_quintis_pdf, fig_tradeoff_pdf]:
        destination = MONOGRAFIA_IMG_DIR / pdf_path.name
        shutil.copy2(pdf_path, destination)
        copied_pdf_names.add(pdf_path.name)

cap4_figure_outputs["copiada_para_monografia"] = cap4_figure_outputs["caminho_pdf"].map(lambda p: Path(p).name in copied_pdf_names)

if copied_pdf_names:
    print("PDFs copiados para o repositório da monografia:")
    for name in sorted(copied_pdf_names):
        print(f"- {name}")
else:
    print("Repositório irmão mba-tcc-monografia/USPSC-img não encontrado; PDFs mantidos apenas em outputs/capitulo4/.")

print("Arquivos em outputs/capitulo4:")
for path in sorted(CAP4_OUTPUT_DIR.glob("cap4_*")):
    print(f"- {path.name}")

cap4_figure_outputs
